In [1]:
import concurrent.futures
import gzip
import itertools
import json
import pathlib
import multiprocessing
import os
import re
import struct
import warnings

from IPython.display import display, Markdown
import numpy
import polars
from tqdm.notebook import tqdm


EVALUATION_PATH = pathlib.Path.cwd()
INPUT_PATH = EVALUATION_PATH / ".." / "output_dataset"

if str((EVALUATION_PATH / "..").resolve()) not in sys.path:
    sys.path.append(str((EVALUATION_PATH / "..").resolve()))


from utils import list_code
from list_scenarios import list_scenarios_full


os.environ["EVALUATION_PATH"] = str(EVALUATION_PATH)
os.environ["INPUT_PATH"] = str(INPUT_PATH)

# Simulate Padding and Random IV

Here, we simulate CoAP Padding and Partial IV encryption. We base this on the merged tables (`*.merged.csv.gz`) generated as intermediates in [`extract_from_pcaps.ipynb`](./extract_from_pcaps.ipynb). Please run this notebook first. 

Since TinyDTLS hardcodes the record epoch and sequence number as the nonce of its AEAD cipher header and aiocoap neither supported [PIV encryption](https://datatracker.ietf.org/doc/draft-tiloca-core-oscore-piv-enc/00/) nor [the padding option](https://www.ietf.org/archive/id/draft-amsuess-core-cachable-oscore-11.html#appendix-B) at the time of evaluation, we artifitially generate this for CoAPS (`coaps`) and Onion OSCORE (`oscore`) as follows.

## Padding

Padding is added by the following rules:

- For "not constrained" and not block-wise, a random number of padding bytes up to 128 bytes is added if the CoAP payload is shorter than 128 bytes and the padding option with at least 1 byte fits.
- For "not constrained" and block-wise transfer, a random number of padding bytes up to the block size is added if the block payload is shorter than the block size and the padding option with at least 1 byte fits.
- For SCHC and non-block-wise, a random number of padding bytes up to the maximum SCHC PDU is added to the last fragment (or just the PDU if unfragmented) if the CoAP payload is shorter than the maximum SCHC PDU and the padding option with at least 1 byte fits.
- For SCHC and block-wise transfer, a random number of padding bytes up to the block size or maximum SCHC PDU (if the block size would overflow) is added to the last fragment (or just the PDU if unfragmented) if the CoAP payload is shorter than the block size and the padding option with at least 1 byte fits.

## Random IV

For DTLS, the nonce is replaced with a random but unique 64-bit number. For OSCORE, the partial IV is replaced with a random number of the same length (emulating encryption according to https://datatracker.ietf.org/doc/draft-tiloca-core-oscore-piv-enc/00/).

In [2]:
schema_overrides = {
    "coap.token": str,
    "coap.opt.uri_host": str,
    "coap.opt.object_security_piv": str,
    "coap.opt.block_number": str,
    "coap.opt.block_mflag": str,
    "coap.opt.block_size": str,
    "oscore.opt.block_number": str,
    "oscore.opt.block_mflag": str,
    "oscore.opt.block_size": str,
}

WORKERS = multiprocessing.cpu_count()

if WORKERS > 32:
    # otherwise we might run into memory problems ;-)
    WORKERS = 32

DTLS_NONCE_OFFSET = 61
DTLS_NONCE_LEN = 8
# (40: IPv6 header, 8: UDP header, 1: CoAP version, type, tkl,
#   1: CoAP code, 2: MID, 2: minimal OSCORE option header)
OSCORE_BASE_OFFSET = 54
# (2: Datagram tag size, 1: Fragment Compressed Number)
FRAG_BASE_OFFSET = 3
MAX_PAD = 128


def get_device_table(scenario):
    routes_path = INPUT_PATH / f"{scenario}.routes.txt"
    device_table = {}
    with routes_path.open() as routes_file:
        df = polars.read_csv(
            routes_file,
            has_header=False,
            separator=" ",
            new_columns=["name", "ipv6", "eth"],
        ).with_columns(
            polars.col("eth").str.decode("hex")
        )
        for row in df.to_dicts():
            device_table[row["eth"]] = row["name"]
    return device_table


def dev_id_from_macs(src_addr: bytes, dst_addr: bytes, device_table: dict[bytes, str], l2_mode: str):
    if l2_mode == "-peer-based":
        return bytes(a ^ b for a, b in zip(src_addr, dst_addr))
    return src_addr if device_table[src_addr] == "client" else dst_addr


def get_schc_rules(scenarios, prot, l2, stp, l2_mode, device_table):
    schc_path = EVALUATION_PATH / "schc"
    rule_name = f"{prot}{l2}-{stp}"

    rev_device_table = {v: k for k, v in device_table.items()}

    if l2_mode == "-min-rules":
        with (schc_path / f"{rule_name}-rules-min.json").open() as rules_file:
            rules = {
                rev_device_table["client"]: {
                    r["RuleID"]: r for r in json.load(rules_file)
                }
            }
    elif l2_mode == "-peer-based":
        rules = {}
        for rules_filename in schc_path.glob(f"{rule_name}-rules-*.json"):
            if rules_filename.name.endswith("-min.json"):
                continue
            else:
                host = re.sub(r".*-rules-(.*)\.json", r"\1", rules_filename.name)
                host_addr = rev_device_table[host]
                client_addr = rev_device_table["client"]
                with rules_filename.open() as rules_file:
                    dev_id = dev_id_from_macs(host_addr, client_addr, True, l2_mode)
                    rules[dev_id] = {
                        r["RuleID"]: r for r in json.load(rules_file)
                    }
    else:
        rules = {}
        with (schc_path / f"{rule_name}-rules.json").open() as rules_file:
            dev_id = rev_device_table["client"]
            rules[dev_id] = {
                r["RuleID"]: r for r in json.load(rules_file)
            }

    return rules


def direction_applies(upstream, direction):
    direction = direction.upper()
    if upstream:
        return direction in ["UP", "BI"]
    return direction in ["DW", "BI"]


def get_bit_offset_from_schc_rule(frame: bytes, dev_id: bytes, upstream: bool, rules: dict[int, dict], search_field=None, token_len=0):
    rule_id = frame[0]
    rule = rules[dev_id][rule_id]
    bit_offset = rule["RuleIDLength"]
    if "Fragmentation" in rule:
        # THIS IS NOT GENERALIZED AND ASSUMES openschc DEFAULT VALUES FOR DTAG_SIZE AND FCN!
        #rule = rules[dev_id][compr_rule_id]
        # assumes 2 bytes DTAG_SIZE and 1 byte FCN
        bit_offset += FRAG_BASE_OFFSET
        compr_rule_id = (
            (
                (frame[bit_offset // 8] << 8) | frame[(bit_offset // 8) + 1]
            ) >> (8 - (bit_offset % 8))
        ) & 0xff
        if compr_rule_id not in rules[dev_id]:
            # This is not a first fragment
            return 0
        rule = rules[dev_id][compr_rule_id]
        bit_offset += rule["RuleIDLength"]
    if "Compression" in rule:
        for field in rule["Compression"]:
            if search_field and field["FID"].upper() == search_field.upper():
                break
            if not direction_applies(upstream, field["DI"]):
                continue
            if isinstance(field["FL"], str) and field["CDA"] not in ["not-sent", "mapping-sent"]:
                if field["FID"].upper() in ["DTLS.HS.CH.COOKIE", "DTLS.HS.SVR.COOKIE"]:
                    field_length = 17 * 8  # cookie of length 16 + 1 byte for length
                elif field["FID"].upper() in ["COAP.TOKEN"]:
                    field_length = token_len * 8
                else:
                    print(field)
            else:
                field_length = field["FL"]
            if field["CDA"] == "value-sent":
                bit_offset += field_length
            elif field["CDA"] == "lsb":
                bit_offset += field_length - field["MO.VAL"]
            elif field["CDA"] == "mapping-sent":
                idx_len = int(numpy.ceil(numpy.log2(len(field["TV"]))))
                bit_offset += idx_len
    return bit_offset


def get_schc_len(bit_offset: int, byte_len: int):
    if (bit_offset % 8):
        return byte_len + 1
    return byte_len


def replace_schc_iv(
    src_addr: bytes,
    dst_addr: bytes,
    frame: bytes,
    device_table: dict[bytes, str],
    rules: dict[int, dict],
    l2_mode: str,
    orig_iv: bytes,
    new_iv: bytes,
    field=None,
    token_len=0,
):
    upstream = device_table[src_addr] == "client"
    dev_id = dev_id_from_macs(src_addr, dst_addr, device_table, l2_mode)
    bit_offset = get_bit_offset_from_schc_rule(frame, dev_id, upstream, rules, field, token_len)
    if bit_offset == 0:
        return frame
    iv_length = get_schc_len(bit_offset, len(new_iv))
    if bit_offset % 8:
        # Mask surrounding bits and shift accordingly
        old_iv = frame[(bit_offset // 8) : ((bit_offset // 8) + iv_length)]
        orig_iv_list = []
        new_iv_list = []
        shift = int(bit_offset % 8)
        for i, (old, (new_upper, new_lower), (orig_upper, orig_lower)) in enumerate(
            zip(old_iv, itertools.pairwise(b"\0" + new_iv + b"\0"), itertools.pairwise(b"\0" + orig_iv + b"\0"))
        ):
            if i == 0:
                new_iv_list.append(
                     (old & ((0xff << (8 - shift)) & 0xff)) | (new_lower >> shift)
                )
                orig_iv_list.append(
                     (old & ((0xff << (8 - shift)) & 0xff)) | (orig_lower >> shift)
                )
            elif i == (len(old_iv) - 1):
                new_iv_list.append(
                    ((new_upper << (8 - shift)) & 0xff) | (old & (0xff >> shift))
                )
                orig_iv_list.append(
                    ((orig_upper << (8 - shift)) & 0xff) | (old & (0xff >> shift))
                )
            else:
                new_iv_list.append(
                    ((new_upper << (8 - shift)) & 0xff) | (new_lower >> shift)
                )
                orig_iv_list.append(
                    ((orig_upper << (8 - shift)) & 0xff) | (orig_lower >> shift)
                )
        if bytes(orig_iv_list) != old_iv:
            # Check if the old iv is actually the original iv, just in case
            # our previous first fragment check in get_bit_offset_from_schc_rule() was
            # a false positive
            return frame
        new_iv = bytes(new_iv_list)
    frame = frame[:(bit_offset // 8)] + new_iv + frame[((bit_offset // 8) + iv_length):]
    return frame


def parse_block_flag(block_flag):
    if block_flag is None:
        return None
    if "," in block_flag:
        if any(f.strip() == "0" for f in block_flag.split(",")):
            return 0
        else:
            return 1
    else:
        return int(block_flag)


def parse_block_exp(block_exp):
    if block_exp is None:
        return None
    if "," in block_exp:
        exps = block_exp.split(",")
        assert all(int(exps[0]) == int(e) for e in exps)
        return int(exps[0])
    else:
        return int(block_exp)
        

def pad_schc(
    frame,
    rules,
    device_table,
    l2_mode,
    src_addr,
    dst_addr,
    coap_payload,
    coap_block_flag,
    coap_block_exp,
    coap_block_payload,
    oscore_block_flag,
    oscore_block_exp,
    oscore_payload,
    max_len,
):
    coap_block_flag = parse_block_flag(coap_block_flag)
    coap_block_exp = parse_block_exp(coap_block_exp)
    oscore_block_flag = parse_block_flag(oscore_block_flag)
    oscore_block_exp = parse_block_exp(oscore_block_exp)
    
    rule_id = frame[0]
    dev_id = dev_id_from_macs(src_addr, dst_addr, device_table, l2_mode)
    rule = rules[dev_id][rule_id]
    if "Fragmentation" in rule:
        bit_offset = rule["RuleIDLength"]
        # assumes DTAG size = 2 and FCN size = 1
        last_frag = frame[bit_offset // 8] & 0b00100000
        if not last_frag:
            # only add padding to last fragment
            return frame
    if (
        (coap_block_flag is not None and coap_block_flag == 0)
        or (oscore_block_flag is not None and oscore_block_flag == 0)
    ):
        # this is the last block

        if coap_payload is None or (oscore_block_flag is not None and oscore_payload is None):
            # this is just a response to a block wise request or
            # a request requesting the next block
            return frame
        
        # see https://www.rfc-editor.org/rfc/rfc7959.html#section-2.2
        if oscore_block_exp:
            exp_block_size = 2**(4 + oscore_block_exp)
        else:
            exp_block_size = 2**(4 + coap_block_exp)
        
        if coap_block_payload is not None:
            payload_length = coap_block_payload
        elif oscore_payload is not None:
            payload_length = oscore_payload
        elif coap_payload is not None:
            payload_length = coap_payload
        else:
            payload_length = 0
        
        # check for exp_block_size - 1 since the padding option needs to be at least 2 byte long
        if payload_length < (exp_block_size - 1):
            if (len(frame) + (exp_block_size - payload_length)) > max_len:
                if (max_len - len(frame) + 1) <= 2:
                    return frame
                pad_len = numpy.random.randint(2, max_len - len(frame) + 1)
            else:
                pad_len = numpy.random.randint(2, (exp_block_size - payload_length) + 1)
        else:
            # There is no padding needed
            return frame
    elif coap_block_flag is not None or oscore_block_flag is not None:
        # is block-wise, but not last block
        return frame
    else:
        # check for max_len - 1 since the padding option needs to be at least 2 byte long
        if len(frame) < (max_len - 1):
            pad_len = numpy.random.randint(2, max_len - len(frame) + 1)
        else:
            # There is no padding needed
            return frame
    # insert before last byte to preserve SCHC padding
    return frame[:-1] + bytes(numpy.random.randint(0, 256) for _ in range(pad_len)) + frame[-1:]


def pad(
    frame,
    coap_payload,
    coap_block_flag,
    coap_block_exp,
    coap_block_payload,
    oscore_block_flag,
    oscore_block_exp,
    oscore_payload,
    max_len,
):
    coap_block_flag = parse_block_flag(coap_block_flag)
    coap_block_exp = parse_block_exp(coap_block_exp)
    oscore_block_flag = parse_block_flag(oscore_block_flag)
    oscore_block_exp = parse_block_exp(oscore_block_exp)
    
    if (
        (coap_block_flag is not None and coap_block_flag == 0)
        or (oscore_block_flag is not None and oscore_block_flag == 0)
    ):
        # this is the last block
        
        if coap_payload is None or (oscore_block_flag is not None and oscore_payload is None):
            # this is just a response to a block wise request or
            # a request requesting the next block
            return frame
            
        # see https://www.rfc-editor.org/rfc/rfc7959.html#section-2.2
        if oscore_block_exp:
            exp_block_size = 2**(4 + oscore_block_exp)
        else:
            exp_block_size = 2**(4 + coap_block_exp)
        
        if coap_block_payload is not None:
            payload_length = coap_block_payload
        elif oscore_payload is not None:
            payload_length = oscore_payload
        elif coap_payload is not None:
            payload_length = coap_payload
        else:
            payload_length = 0
        
        # check for exp_block_size - 1 since the padding option needs to be at least 2 byte long
        if payload_length < (exp_block_size - 1):
            if (len(frame) + (exp_block_size - payload_length)) > max_len:
                if (max_len - len(frame) + 1) <= 2:
                    return frame
                pad_len = numpy.random.randint(2, max_len - len(frame) + 1)
            else:
                pad_len = numpy.random.randint(2, (exp_block_size - payload_length) + 1)
        else:
            # There is no padding needed
            return frame
    elif coap_block_flag is not None or oscore_block_flag is not None:
        # is block-wise, but not last block
        return frame
    else:
        if oscore_payload is not None:
            payload_length = oscore_payload
        elif coap_payload is not None:
            payload_length = coap_payload
        else:
            payload_length = 0
        # check for max_len - 1 since the padding option needs to be at least 2 byte long
        if payload_length < (MAX_PAD - 1):
            pad_len = numpy.random.randint(2, MAX_PAD - payload_length + 1)
        else:
            # There is no padding needed
            return frame
    return frame + bytes(numpy.random.randint(0, 256) for _ in range(pad_len))


def pad_and_replace_with_randiv(scenario, prot, l2, stp, l2_mode, blk):
    rand_sink_filename = INPUT_PATH / f"{scenario}_randiv_pad.training.csv.gz"
    if rand_sink_filename.exists():
        return
    lf = polars.scan_csv(INPUT_PATH / f"{scenario}.merged.csv.gz", separator=";", schema_overrides=schema_overrides)
    max_len = lf.select("eth.payload").with_columns(
        polars.col("eth.payload").str.len_chars()
    ).max().collect()[0, 0]
    if l2:
        device_table = get_device_table(scenario)
        rules = get_schc_rules(scenario, prot, l2, stp, l2_mode, device_table)
    if prot == "coaps":
        frames = lf.select("eth.payload").count().collect()[0, 0]
        nonce = numpy.random.randint(0, (1 << 64), dtype=numpy.uint64)
        max_step = (1 << 64) // frames
        nonces = []
        for i in range(frames):
            nonces.append(struct.pack("!Q", nonce).hex())
            with warnings.catch_warnings(
                action="ignore",
                category=RuntimeWarning,
            ):
                nonce = nonce + numpy.random.randint(1, max_step, dtype=numpy.uint64)
        numpy.random.shuffle(nonces)
        lf = lf.with_columns(
            new_nonces=polars.Series("new_nonces", nonces),
        )
        if l2:
            training = lf.with_columns(
                polars.col("eth.src").str.replace_all(":", "").str.decode("hex"),
                polars.col("eth.dst").str.replace_all(":", "").str.decode("hex"),
                polars.col("eth.payload").str.decode("hex"),
                polars.col("new_nonces").str.decode("hex"),
            ).with_columns(
                polars.struct(
                    "eth.src",
                    "eth.dst",
                    "eth.payload",
                    "dtls.record.epoch",
                    "dtls.record.sequence_number",
                    "new_nonces",
                ).map_elements(
                    lambda s: replace_schc_iv(
                        s["eth.src"],
                        s["eth.dst"],
                        s["eth.payload"],
                        device_table,
                        rules,
                        l2_mode,
                        bytes.fromhex(
                            f"{s["dtls.record.epoch"]:04x}{s["dtls.record.sequence_number"]:012x}"
                        ),
                        s["new_nonces"],
                    ),
                    return_dtype=polars.Binary,
                ).alias("eth.payload")
            ).with_columns(
                polars.struct(
                    "eth.src",
                    "eth.dst",
                    "eth.payload",
                    "coap.block_length",
                    "coap.opt.block_mflag",
                    "coap.opt.block_size",
                    "coap.payload_length",
                ).map_elements(
                    lambda s: pad_schc(
                        s["eth.payload"],
                        rules,
                        device_table,
                        l2_mode,
                        s["eth.src"],
                        s["eth.dst"],
                        s["coap.payload_length"],
                        s["coap.opt.block_mflag"],
                        s["coap.opt.block_size"],
                        s["coap.block_length"],
                        None,
                        None,
                        None,
                        max_len // 2,
                    ),
                    return_dtype=polars.Binary,
                ).bin.encode("hex").alias("eth.payload")
            )
        else:
            training = lf.with_columns(
                polars.concat_str(
                    [
                        # double offset and length since we have hex dumps here
                        polars.col("eth.payload").str.slice(0, 2 * DTLS_NONCE_OFFSET),
                        polars.col("new_nonces"),
                        polars.col("eth.payload").str.slice(2 * (DTLS_NONCE_OFFSET + DTLS_NONCE_LEN))
                    ]
                ).alias("eth.payload")
            ).with_columns(
                polars.struct(
                    "eth.payload",
                    "coap.block_length",
                    "coap.opt.block_mflag",
                    "coap.opt.block_size",
                    "coap.payload_length",
                ).map_elements(
                    lambda s: pad(
                        bytes.fromhex(s["eth.payload"]),
                        s["coap.payload_length"],
                        s["coap.opt.block_mflag"],
                        s["coap.opt.block_size"],
                        s["coap.block_length"],
                        None,
                        None,
                        None,
                        max_len // 2,
                    ),
                    return_dtype=polars.Binary,
                ).bin.encode("hex").alias("eth.payload")
            )
        del nonces
    elif prot == "oscore":
        if l2:
            training = lf.with_columns(
                has_piv=~polars.col("coap.opt.object_security_piv").is_null(),
                piv_len=polars.col("coap.opt.object_security_piv").str.len_chars() // 2,
                token_len=polars.col("coap.token").str.len_chars() // 2,
            ).with_columns(
                polars.col("eth.src").str.replace_all(":", "").str.decode("hex"),
                polars.col("eth.dst").str.replace_all(":", "").str.decode("hex"),
                polars.col("eth.payload").str.decode("hex"),
                polars.col("coap.opt.object_security_piv").str.decode("hex"),
            ).with_columns(
                polars.struct(
                    "eth.src",
                    "eth.dst",
                    "eth.payload",
                    "coap.opt.object_security_piv",
                    "has_piv",
                    "piv_len",
                    "token_len",
                ).map_elements(
                    lambda s: replace_schc_iv(
                        s["eth.src"],
                        s["eth.dst"],
                        s["eth.payload"],
                        device_table,
                        rules,
                        l2_mode,
                        s["coap.opt.object_security_piv"],
                        bytes(
                            numpy.random.randint(0, 256) for _ in range(s["piv_len"])
                        ),
                        field="COAP.OSCORE.PIV",
                        token_len=s["token_len"],
                    ) if s["has_piv"] else s["eth.payload"],
                    return_dtype=polars.Binary,
                ).alias("eth.payload")
            ).with_columns(
                polars.struct(
                    "eth.src",
                    "eth.dst",
                    "eth.payload",
                    "coap.block_length",
                    "coap.opt.block_mflag",
                    "coap.opt.block_size",
                    "coap.payload_length",
                    "oscore.opt.block_mflag",
                    "oscore.opt.block_size",
                    "oscore.payload_length",
                ).map_elements(
                    lambda s: pad_schc(
                        s["eth.payload"],
                        rules,
                        device_table,
                        l2_mode,
                        s["eth.src"],
                        s["eth.dst"],
                        s["coap.payload_length"],
                        s["coap.opt.block_mflag"],
                        s["coap.opt.block_size"],
                        s["coap.block_length"],
                        s["oscore.opt.block_mflag"],
                        s["oscore.opt.block_size"],
                        s["oscore.payload_length"],
                        max_len // 2,
                    ),
                    return_dtype=polars.Binary,
                ).bin.encode("hex").alias("eth.payload")
            )
        else:
            display(scenario)
            training = lf.with_columns(
                has_piv=~polars.col("coap.opt.object_security_piv").is_null(),
                piv_len=polars.col("coap.opt.object_security_piv").str.len_chars() // 2,
                token_len=polars.col("coap.token").str.len_chars() // 2,
                uri_host_len=polars.when(
                    polars.col("coap.opt.uri_host").str.len_chars() > 0
                ).then(
                    polars.col("coap.opt.uri_host").str.len_chars() + 1
                ).otherwise(0),
            ).with_columns(
                piv_offset = OSCORE_BASE_OFFSET + polars.col("token_len") + polars.col("uri_host_len")
            ).with_columns(
                polars.struct(
                    "eth.payload",
                    "has_piv",
                    "piv_offset",
                    "piv_len",
                    "coap.opt.object_security_piv"
                ).map_elements(
                    lambda s: s["eth.payload"][
                        : (2 * s["piv_offset"])
                    ] + "".join(
                        f"{numpy.random.randint(0, 256):02x}" for _ in range(s["piv_len"])
                    ) + s["eth.payload"][
                        (2 * (s["piv_offset"] + s["piv_len"])):
                    ] if s["has_piv"] else s["eth.payload"],
                    return_dtype=polars.String,
                ).alias("eth.payload")
            ).with_columns(
                polars.struct(
                    "eth.payload",
                    "coap.block_length",
                    "coap.opt.block_mflag",
                    "coap.opt.block_size",
                    "coap.payload_length",
                    "oscore.opt.block_mflag",
                    "oscore.opt.block_size",
                    "oscore.payload_length",
                ).map_elements(
                    lambda s: pad(
                        bytes.fromhex(s["eth.payload"]),
                        s["coap.payload_length"],
                        s["coap.opt.block_mflag"],
                        s["coap.opt.block_size"],
                        s["coap.block_length"],
                        s["oscore.opt.block_mflag"],
                        s["oscore.opt.block_size"],
                        s["oscore.payload_length"],
                        max_len // 2,
                    ),
                    return_dtype=polars.Binary,
                ).bin.encode("hex").alias("eth.payload")
            )
    else:
        display(scenario)
        raise RuntimeError()
    with gzip.open(rand_sink_filename, "wb") as csvfile:
        training.select(
            ["frame.number", "frame.time_epoch", "client.type", "eth.payload"]
        ).with_columns(
            **{
                "eth.payload": polars.col("eth.payload").str.pad_end(max_len, "x")
            }
        ).sink_csv(csvfile, separator=";")


print(f"Replace with random IV and pad on {WORKERS} workers")
with concurrent.futures.ProcessPoolExecutor(max_workers=WORKERS) as executor:
    def _pad_and_replace_with_randiv(args):
        scenario, prot, l2, stp, l2_mode, _, _, blk, _ = args
        pad_and_replace_with_randiv(scenario, prot, l2, stp, l2_mode, blk)

    scenarios = list(list_scenarios_full(filter_protocol=["coaps", "oscore"], filter_randiv_pad=True))
    with tqdm(
        executor.map(
            _pad_and_replace_with_randiv,
            scenarios,
        ),
        total=len(scenarios),
    ) as progress:
        for _ in progress:
            continue

Replace with random IV and pad on 24 workers


  0%|          | 0/160 [00:00<?, ?it/s]